In [1]:
import config
import pandas as pd
import modules.capiq as capiq
import modules.gpt as gpt
import textwrap
import modules.sql as sql

In [2]:
excel_path = f'{config.ROOT}/data/testset.xlsx'

db_path = f'{config.ROOT}/data/testset.db'
table_name = 'prompt_outputs'

In [3]:
# Load capiq dataset
df = pd.read_feather(f'{config.ROOT}/data/transcript-detail.feather')

# Keep uhaul calls:
df = df[df['companyid'] == 345588]

In [4]:
# Specify the list id_list of transcript ids to be examined
id_list = sql.get_transcript_ids(db_path, table_name)
print(f'Number of transcripts in the database: {len(id_list)}')



Number of transcripts in the database: 20


In [5]:
# Get transcripts
transcript_text = capiq.get_transcripts(transcriptids=id_list, limit=1e7)

Loading library list...
Done
WRDS connection established.


In [6]:
assert len(id_list) == len(transcript_text), "Length of id_list and transcript_text do not match"
print(f"Length of id_list: {len(id_list)}")
print(f"Length of transcript_text: {len(transcript_text)}")

Length of id_list: 20
Length of transcript_text: 20


In [7]:
prompt_name = 'SimpleScore'

# Get GPT response:
gpt_response_output = gpt.get_responses(prompt_name, id_list)

In [8]:
print(gpt_response_output)

{9064: {'score': 20, 'reasoning': "The transcript does not provide clear evidence of collusive intent. While there are discussions about pricing strategies and market conditions, the language used is more focused on competitive positioning and operational strategies rather than any explicit agreement or coordination with competitors. The mention of communicating price increases and observing competitors' pricing behavior could suggest a level of market awareness, but it does not indicate collusion. Overall, the tone and content of the call reflect a company navigating a challenging market rather than conspiring with others."}, 15056: {'score': 20, 'reasoning': 'The transcript does not provide clear evidence of collusive intent. While there are discussions about pricing pressures and competition, the executives emphasize maintaining their pricing strategy and not engaging in destructive pricing behaviors. They express a desire for competitors to maintain prices as well, which could sugg

In [4]:
# Make prompt_variables as a list of the keys of the first element of gpt_response_output
response_format_class = gpt.prompts[prompt_name]['response_format']
prompt_variables = list(response_format_class.__fields__.keys())
print(prompt_variables)

['score', 'reasoning']


/var/folders/38/klt_lxb9043crs1v81qnycr40000gn/T/ipykernel_81082/2077057283.py:4: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  prompt_variables = list(response_format_class.__fields__.keys())


In [10]:
# Add columns for current prompt to the database
sql.add_columns_for_prompt(db_path, table_name, prompt_name, prompt_variables)

In [11]:
# Populate the database with the responses
sql.populate_columns_from_response(db_path, table_name, prompt_name, gpt_response_output)

In [5]:
# This just prints the columns and the contents of the database for quick inspection

import sqlite3

# Connect to the SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Query the table schema to see the columns
cursor.execute(f"PRAGMA table_info({table_name})")
columns = cursor.fetchall()

# Print the columns
for column in columns:
    print(column)

# Optionally, query the table data to see the new columns
cursor.execute(f"SELECT * FROM {table_name} LIMIT 5")
rows = cursor.fetchall()

# Print the table data
for row in rows:
    print(row)

# Close the connection
conn.close()



(0, 'companyid', 'INT', 0, None, 0)
(1, 'transcriptid', 'INT', 0, None, 0)
(2, 'headline', 'TEXT', 0, None, 0)
(3, 'mostimportantdateutc', 'NUM', 0, None, 0)
(4, 'keydeveventtypename', 'TEXT', 0, None, 0)
(5, 'companyname', 'TEXT', 0, None, 0)
(6, 'joe_score', 'INT', 0, None, 0)
(7, 'joe_comment', 'TEXT', 0, None, 0)
(8, 'SimpleScore_score_1', 'TEXT', 0, None, 0)
(9, 'SimpleScore_reasoning_1', 'TEXT', 0, None, 0)
(10, 'SimpleScore_score_2', 'TEXT', 0, None, 0)
(11, 'SimpleScore_reasoning_2', 'TEXT', 0, None, 0)
(345588, 9064, 'AMERCO, Q4 2008 Earnings Call, Jun-05-2008', '2008-06-05 00:00:00', 'Earnings Calls', 'U-Haul Holding Company', 90, 'This gets a really high score because it is referring to it and a rival keeping price above cost and that there is the expectation of the rival matching them. It is also stating how it is communicating to the rival. ', '20', "The transcript does not provide clear evidence of collusive intent. While there are discussions about pricing strategies and

In [6]:
# Export the database for inspection if necessary
csv_path = f'{config.ROOT}/data/testset.csv'
sql.export_table_to_csv(db_path, table_name, csv_path)

In [10]:
losses = sql.calculate_loss(db_path, table_name)
# For each prompt in losses, print the loss
print('Losses: ')
for prompt, loss in losses.items():
    print(f'{prompt}: {loss}')

Losses: 
SimpleScore_1: 532.5
SimpleScore_2: 547.5
